### Convert generated opt output coordinates into spe input files of ligand and optimized complex

In [1]:
# Workflow
# Extract coordinates within opt file
# Convert into spe input file for M06/def2tzvp/Lanl2dz calculationfrom xyz2graph import MolGraph, to_networkx_graph, to_plotly_figure


In [2]:
# Install a pip package in the current Jupyter kernel
import sys
!{sys.executable} -m pip install git+https://github.com/zotko/xyz2graph.git
# python -m pip install git+https://github.com/zotko/xyz2graph.git

  Cloning https://github.com/zotko/xyz2graph.git to c:\users\george\appdata\local\temp\pip-req-build-70lnqgu6
  Resolved https://github.com/zotko/xyz2graph.git to commit b11d841e60432225740620aca1582bcdc562f115
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'


  Running command git clone --filter=blob:none --quiet https://github.com/zotko/xyz2graph.git 'C:\Users\George\AppData\Local\Temp\pip-req-build-70lnqgu6'


In [3]:
import os
from pathlib import Path
import re #Import RegEx
import os
import math
import numpy as np
import pandas as pd
from pathlib import Path
from plotly.offline import offline
import networkx as nx
from xyz2graph import MolGraph, to_networkx_graph, to_plotly_figure

In [6]:
directory = os.getcwd()
directory

'C:\\Users\\George\\Desktop\\Research_UCLA\\CIC\\Computational Work\\Project cluster version 2\\project_catalyst_repurposing\\structures_ligand_id'

In [5]:
# Check if all three files (opt, spe-complex and spe-ligand is present for each ligand)

opt_directory = directory + '\\pd_me2_structures\\opt'
spe_complex_directory = directory + '\\pd_me2_structures\\spe'
complex_xyz_directory = directory + '\\pd_me2_structures\\complex_xyz'
ligand_xyz_directory = directory + '\\pd_me2_structures\\ligand_xyz'
directory_list = [opt_directory, spe_complex_directory]
replace_list = ['-opt','-spe']

opt_list = []
spe_complex_list =[]
name_lists = [opt_list,spe_complex_list]

for check_directory,word,name_list in zip(directory_list,replace_list,name_lists):
# Read in .out file
    index_count = 0
#     name_list = []
    for subdir,dirs,files in os.walk(check_directory):                  # Loop over each directory, subdirectory and files
        for file in files:                                      # Loop over each file
            if any([file.endswith('.out')]):                    # If file is a .out file
                filename = os.path.join(subdir, file)       # Return path to file
                name = Path(filename).stem           # Extract filename from the end of path and return as a string
                name = name.replace(word,'')     #Remove -opt in string to get just the name of ligand 'pd-x-x-x'
                name_list.append(name)
                
                
print('Number of files in opt: ', len(opt_list))
print('Number of files in spe: ', len(spe_complex_list))
missing_in_opt_list = np.unique([x for x in spe_complex_list if x not in opt_list])
missing_in_spe_complex_list = np.unique([x for x in opt_list if x not in spe_complex_list])


print('Missing in opt list: ', missing_in_opt_list)
print('Missing in spe complex list: ',missing_in_spe_complex_list)

missing_list = []
missing_list.extend(missing_in_opt_list)
missing_list.extend(missing_in_spe_complex_list)
print(missing_list)

# with open("missing_list.txt", "w") as output:
#     output.write(str(missing_list))

Number of files in opt:  490
Number of files in spe:  490
Missing in opt list:  []
Missing in spe complex list:  []
[]


In [7]:
#Define functions

def get_h_atoms(cart_distances):
    sorted_indices = sorted(range(len(cart_distances)), key=lambda i: cart_distances[i])
    second_smallest_index = sorted_indices[1]
    third_smallest_index = sorted_indices[2]
    fourth_smallest_index = sorted_indices[3]
    fifth_smallest_index = sorted_indices[4] 
    return second_smallest_index, third_smallest_index, fourth_smallest_index, fifth_smallest_index

def get_donor_atoms(cart_distances, element_list):
    # Sort the indices based on cartesian distances
    sorted_indices = sorted(range(len(cart_distances)), key=lambda i: cart_distances[i])
    
    # Filter out indices where the corresponding element is 'H'
    filtered_indices = [i for i in sorted_indices if element_list[i] != 'H']

    second_smallest_index = filtered_indices[1]
    third_smallest_index = filtered_indices[2]
    fourth_smallest_index = filtered_indices[3]
    fifth_smallest_index = filtered_indices[4]
    
    return second_smallest_index, third_smallest_index, fourth_smallest_index, fifth_smallest_index

def get_neighbor_atoms(numbers):
    sorted_indices = sorted(range(len(numbers)), key=lambda i: numbers[i]) 
    return sorted_indices


def xyz_to_dataframe(filename):
    with open(filename, 'r') as file:
        lines = file.readlines()
        
    # Initialize lists to store data
    atoms = []
    x_coords = []
    y_coords = []
    z_coords = []
    
    # Parse the lines and extract coordinates
    for line in lines:
        parts = line.split()
        atoms.append(parts[0])
        x_coords.append(float(parts[1]))
        y_coords.append(float(parts[2]))
        z_coords.append(float(parts[3]))

    # Create a pandas DataFrame
    data = {
        'Atom': atoms,
        'X': x_coords,
        'Y': y_coords,
        'Z': z_coords
    }

    xyz_df = pd.DataFrame(data)    
    return(xyz_df)

# def generate_ligand_spe(atoms_to_remove,ligand_name):


# Define element dictionary once, globally
element_dict = {
    "1": "H",   "2": "He",  "3": "Li",  "4": "Be",  "5": "B",
    "6": "C",   "7": "N",   "8": "O",   "9": "F",   "10": "Ne",
    "11": "Na", "12": "Mg", "13": "Al", "14": "Si", "15": "P",
    "16": "S",  "17": "Cl", "18": "Ar", "19": "K",  "20": "Ca",
    "21": "Sc", "22": "Ti", "23": "V",  "24": "Cr", "25": "Mn",
    "26": "Fe", "27": "Co", "28": "Ni", "29": "Cu", "30": "Zn",
    "31": "Ga", "32": "Ge", "33": "As", "34": "Se", "35": "Br",
    "36": "Kr", "37": "Rb", "38": "Sr", "39": "Y",  "40": "Zr",
    "41": "Nb", "42": "Mo", "43": "Tc", "44": "Ru", "45": "Rh",
    "46": "Pd", "47": "Ag", "48": "Cd", "49": "In", "50": "Sn",
    "51": "Sb", "52": "Te", "53": "I",  "54": "Xe", "55": "Cs",
    "56": "Ba", "57": "La", "58": "Ce", "59": "Pr", "60": "Nd",
    "61": "Pm", "62": "Sm", "63": "Eu", "64": "Gd", "65": "Tb",
    "66": "Dy", "67": "Ho", "68": "Er", "69": "Tm", "70": "Yb",
    "71": "Lu", "72": "Hf", "73": "Ta", "74": "W",  "75": "Re",
    "76": "Os", "77": "Ir", "78": "Pt", "79": "Au", "80": "Hg",
    "81": "Tl", "82": "Pb", "83": "Bi", "84": "Po", "85": "At",
    "86": "Rn", "87": "Fr", "88": "Ra", "89": "Ac", "90": "Th",
    "91": "Pa", "92": "U",  "93": "Np", "94": "Pu", "95": "Am",
    "96": "Cm", "97": "Bk", "98": "Cf", "99": "Es", "100": "Fm",
    "101": "Md","102": "No", "103": "Lr","104": "Rf","105": "Db",
    "106": "Sg","107": "Bh", "108": "Hs","109": "Mt","110": "Ds",
    "111": "Rg","112": "Cn","113": "Nh","114": "Fl","115": "Mc",
    "116": "Lv","117": "Ts","118": "Og"
}


### Obtain XYZ coordinates, obtain metal and  ligand atom labels from -opt.out files.
### Ligand atom labels for each bond distance away from the metal center is also obtained. 
### Skip this code to only generate spe.com files from opt.out files

In [8]:
xyz_match = ['X           Y           Z']
nbo_match = ['Natural Population Analysis']
unique_atom_list = []
ligand_atom_pair_list = []
missing_nbo_calc = []

# Intialize list to store atom labels
names = []
pd_atoms = []
methyl_atom_1 = []
methyl_atom_2 = []
hydrogens_methyl_1 = []
hydrogens_methyl_2 = []
ligand_atom_1 = []
ligand_atom_2 = []
atom_distance_lists_2 = []
atom_distance_lists_3 = []
atom_distance_lists_4 = []
atom_distance_lists_2_a = []
atom_distance_lists_3_a = []
atom_distance_lists_4_a = []
atom_distance_lists_2_b = []
atom_distance_lists_3_b = []
atom_distance_lists_4_b = []
shortest_path_atom_label_lists = []

for subdir,dirs,files in os.walk(opt_directory):                  # Loop over each directory, subdirectory and files
    for file in files:                                      # Loop over each file
        if any([file.endswith('-opt.out')]):                    # If file is a .out file
            filename = os.path.join(subdir, file)       # Return path to file
            name = Path(filename).stem.replace('-opt',"")         # Extract filename from the end of path and return as a string
            print(name)
        
            mylines = []
            with open (filename, 'rt') as myfile:       # Open .out for reading text
                # myfile = myfile.read()                # Read the entire file to a string
                for myline in myfile:                    # For each line, stored as myline,
                    mylines.append(myline)               # add its contents to mylines list.
                    
#                 # Find XYZ Coordinates
                for line in mylines:
                    if 'NAtoms=' in line:
                        number_list = re.findall('-?\d*\.?\d+',line)            # get NAtoms value
                        natoms = int(number_list[0])
#                         print(natoms)                
                
                xyz_count = 0
                for line in mylines:
                    for phrase in xyz_match:                                # iterate through each phrases
                        if phrase in line:                                          # check if phrase is in line
                            xyz_count = xyz_count + 1
                
                line_count = 0
                for line in mylines:
                    line_count = line_count + 1
                    for phrase in xyz_match:                                # iterate through each phrases
                        if phrase in line:                                          # check if phrase is in line
                            xyz_count = xyz_count - 1
                            if xyz_count > 0:
                                continue
                            elif xyz_count == 0:
                                
                                                                               # For loop for generating the XYZ coordinates
                                count = 0
                                xyz = []
                                while count < natoms:
                                    count = count + 1
                                    xyz.append(mylines[line_count + 1])
                                    line_count = line_count +1

                x_coord = []
                y_coord = []
                z_coord = []
                atom_symbol =[]
                
                # Generate XYZ file in .txt form, then find xyz coordinates for metal, atom_a and atom_b
                for line in xyz:
                    number_list = re.findall('-?\d*\.?\d+',line)
                    atom_number = int(number_list[0])
                    element_number = int(number_list[1])
                    atom_x = float(number_list[3])
                    atom_x = f"{atom_x:.6f}"
                    atom_y = float(number_list[4])
                    atom_y = f"{atom_y:.6f}"
                    atom_z = float(number_list[5])
                    atom_z = f"{atom_z:.6f}"
                
                    # Make xyz coord into .txt file 
                    x_coord.append(atom_x)
                    y_coord.append(atom_y)
                    z_coord.append(atom_z)
                    atom_symbol.append(element_dict[str(element_number)])
                    unique_atoms = list(set(atom_symbol))
                    name_xyz = name + '-complex_xyz.txt'            
                    
                data = {
                    'Atom': atom_symbol,
                    'X': x_coord,
                    'Y': y_coord,
                    'Z': z_coord
                }

                xyz_df = pd.DataFrame(data)

                # Check if output directory for complex xyz exist, create if it doesn't
                os.makedirs(complex_xyz_directory, exist_ok=True)

                # Combine the directory and filename to create the full file path
                complex_xyz_file_path = os.path.join(complex_xyz_directory, name_xyz)

                
#               xyz_df.to_csv(name_xyz, header=False, index=False, sep = " ")          # Generates .txt file                
                new_row = pd.DataFrame({'Atom':'filler', 'X':'', 'Y':'', 'Z':''}, index=[0])
                xyz_df_txt = pd.concat([new_row,xyz_df.loc[:]]).reset_index(drop=True) 
                new_row_2 = pd.DataFrame({'Atom':natoms, 'X':'', 'Y':'', 'Z':''}, index=[0])
                xyz_df_txt = pd.concat([new_row_2,xyz_df_txt.loc[:]]).reset_index(drop=True)

                # Save complex_xyz as text file in the complex_xyz directory            
                xyz_df_txt.to_csv(complex_xyz_file_path, header=False, index=False, sep = " ")          # Generates .txt file   

                                
                # Identify ligand atoms from xyz_df
                condition = (xyz_df['Atom'] == 'Pd')
                pd_row = xyz_df[condition].iloc[0]
                pd_atom = xyz_df[condition].index[0]
                print('Pd number index: ', pd_atom)


                pd_x = pd_row['X']
                pd_y = pd_row['Y']
                pd_z = pd_row['Z']

                #     print(pd_x,pd_y,pd_z)

                cart_distance_pd_list = []
                element_list = []
                for index, row in xyz_df.iterrows():           #Obtain difference between Pd(x,y,z) coordinates and the rest of the atom's cartesian coordinates

                    x_diff = float(pd_x) - float(row['X'])
                    y_diff = float(pd_y) - float(row['Y'])
                    z_diff = float(pd_z) - float(row['Z'])
                    element = row['Atom']
                    
                    cart_distance_pd = np.sqrt(x_diff**2 + y_diff**2 + z_diff**2)

                    cart_distance_pd_list.append(cart_distance_pd)
                    element_list.append(element)
                print(element_list)

                donor_atom_1, donor_atom_2, donor_atom_3, donor_atom_4 = get_donor_atoms(cart_distance_pd_list,element_list)   #Get atoms coordinated to Pd
                
                # pd_row = xyz_df[condition].iloc[0]    

                donor_atoms = [donor_atom_1, donor_atom_2, donor_atom_3, donor_atom_4]   # List of atom indices that is bonded to Pd

                new_donor_atoms = []

                for donor_atom in donor_atoms:
            #         print('Atom index closest to Pd: ', donor_atom)
                    atom_name = xyz_df.iloc[donor_atom,0]
                    if atom_name == 'C':
                        new_donor_atoms.append(donor_atom)   #Obtain index with carbon atom only

                donor_atom_df = xyz_df.loc[new_donor_atoms]

                #Identifying methyl groups attached on Pd
                methyl_atoms = []
                hydrogen_atoms = []
                hydrogen_indices = []

                for donor_atom in new_donor_atoms:    #iterate through the index of each carbon donor atom

                    c_x = xyz_df.iloc[donor_atom,1]   #obtain x y z coordinates for each carbon donor atom
                    c_y = xyz_df.iloc[donor_atom,2]
                    c_z = xyz_df.iloc[donor_atom,3]

                    cart_distance_c_list = []
                    for index, row in xyz_df.iterrows():
                        x_diff = float(c_x) - float(row['X'])
                        y_diff = float(c_y) - float(row['Y'])
                        z_diff = float(c_z) - float(row['Z'])            

                        cart_distance_c = np.sqrt(x_diff**2 + y_diff**2 + z_diff**2)
                        cart_distance_c_list.append(cart_distance_c)

                    h_atom_1, h_atom_2, h_atom_3, h_atom_4 = get_h_atoms(cart_distance_c_list)

                    h_atoms = [h_atom_1, h_atom_2, h_atom_3, h_atom_4]
            #         print(donor_atom, h_atoms)

                    new_h_atoms = []
                    new_h_indices = []

                    for h_atom in h_atoms:
                        atom_name = xyz_df.iloc[h_atom,0]
                        if atom_name == 'H':
                            h_index = h_atom
                            new_h_indices.append(h_index)
                            h_atom = h_atom + 1
                            new_h_atoms.append(h_atom)

            #         print('Index of donor atom: ', donor_atom, 'Number of Hs: ', len(new_h_atoms)) 

                    if len(new_h_atoms) == 3:
                        methyl_atoms.append(donor_atom)
                        hydrogen_atoms.append(new_h_atoms)
                        hydrogen_indices.append(new_h_indices)


            #     print('Methyl atom indices: ', methyl_atoms)
            #     print('Hydrogen atom indices: ', hydrogen_atoms)

                ligand_atoms_label = []
                new_ligand_atoms_label = []
                ligand_atoms_indicies = []


                ligand_atoms_indicies = [x for x in donor_atoms if x not in methyl_atoms] # Remove atom label to only obtain indices related to ligand atom
                for atom in ligand_atoms_indicies:
                    ligand_atoms_label.append(atom + 1)
                # print('Ligand atom label list: ', ligand_atoms_label)
                ligand_atom_pair_list.append(ligand_atoms_label)
#--------------------------------------------------------------------------------------------------------------------------  

#### Identify ligand atoms based on nbo charge calculated in opt file

                for phrase in nbo_match:                                       # iterate through each phrases
                    nbo_match_count = 0
                    for line in mylines:
                        if phrase in line:                                          # check if phrase is in line
                            nbo_match_count = nbo_match_count + 1
                            count = 0
                            nbo_xyz = []
                            line_number = mylines.index(line) + 6
                            while count < natoms:
                                count = count + 1
                                nbo_xyz.append(mylines[line_number])
                                line_number = line_number + 1

                            with open('nbo.txt', 'w') as filehandle:                   # Save nbo_xyz  into a txt file
                                for listitem in nbo_xyz:
                                    filehandle.write(listitem)

                # Extract NBO charge for metal, atom A and atom B, atom at distance 2,3,4
                if nbo_match_count == 0:
                    print('No NBO Charge calculation for ', name)
                    missing_nbo_calc.append(name)
                    continue


                elif nbo_match_count > 0:    
                    for line in nbo_xyz:
                        number_list = re.findall('-?\d*\.?\d+',line)
                        atom_number = int(number_list[0])

                        if atom_number == ligand_atoms_label[0]:
                            a_nbo = float(number_list[1])
                            print('Ligand atom label: ', atom_number,
                                  ' Ligand atom A nbo: ', a_nbo)

                        elif atom_number == ligand_atoms_label[1]:
                            b_nbo = float(number_list[1]) 
                            print('Ligand atom label: ', atom_number,
                                  ' Ligand atom B nbo: ', b_nbo)                
                
                
                if a_nbo >= b_nbo:
                    new_ligand_atom_a = ligand_atoms_label[0]
                    new_ligand_atom_b = ligand_atoms_label[1]
                elif a_nbo < b_nbo:
                    new_ligand_atom_a = ligand_atoms_label[1]
                    new_ligand_atom_b = ligand_atoms_label[0]
                
                new_ligand_atoms_label.append(new_ligand_atom_a)
                new_ligand_atoms_label.append(new_ligand_atom_b)
                
                print('Atom_a: ', new_ligand_atom_a, 'Atom_b: ', new_ligand_atom_b)
              
                
#--------------------------------------------------------------------------------------------------------------------------                

                    
                #Obtain xyz coordinates for donor atoms
        
                ligand_atom_1_row = xyz_df.loc[new_ligand_atoms_label[0]-1]
                ligand_atom_2_row = xyz_df.loc[new_ligand_atoms_label[1]-1]
                # print('Ligand number index: ', ligand_atom_1_row)

                ligand_atom_1_x = ligand_atom_1_row['X']
                ligand_atom_1_y = ligand_atom_1_row['Y']
                ligand_atom_1_z = ligand_atom_1_row['Z']
                # print('Atom_a_x: ',ligand_atom_1_x)
                # print('Atom_a_y: ',ligand_atom_1_y)                
                # print('Atom_a_z: ',ligand_atom_1_z)                
                
                ligand_atom_2_x = ligand_atom_2_row['X']
                ligand_atom_2_y = ligand_atom_2_row['Y']
                ligand_atom_2_z = ligand_atom_2_row['Z'] 
                # print('Atom_b_x: ',ligand_atom_2_x)
                # print('Atom_b_y: ',ligand_atom_2_y)                
                # print('Atom_b_z: ',ligand_atom_2_z)   
                
                
                #Find atoms at distance 2 (connected to donor atoms)
                
                atom_distance_2 = []
                atom_distance_2_with_a = []
                atom_distance_2_with_b = []
                
                
                for index, row in xyz_df.iterrows():

                    ligand_1_x_diff = float(ligand_atom_1_x) - float(row['X'])
                    ligand_1_y_diff = float(ligand_atom_1_y) - float(row['Y'])
                    ligand_1_z_diff = float(ligand_atom_1_z) - float(row['Z'])

                    ligand_2_x_diff = float(ligand_atom_2_x) - float(row['X'])
                    ligand_2_y_diff = float(ligand_atom_2_y) - float(row['Y'])
                    ligand_2_z_diff = float(ligand_atom_2_z) - float(row['Z'])


                    cart_distance_ligand_1 = np.sqrt(ligand_1_x_diff**2 + ligand_1_y_diff**2 + ligand_1_z_diff**2)
                    if 0.1 < cart_distance_ligand_1 < 1.93 and index != pd_atom:
                        atom_distance_2.append(index+1)
                        atom_distance_2_with_a.append(index+1)


                    cart_distance_ligand_2 = np.sqrt(ligand_2_x_diff**2 + ligand_2_y_diff**2 + ligand_2_z_diff**2)
                    if 0.1 < cart_distance_ligand_2 < 1.93 and index != pd_atom:
                        atom_distance_2.append(index+1)    
                        atom_distance_2_with_b.append(index+1)

                    # Remove duplicates
                atom_distance_2 = list(set(atom_distance_2))
                # print('Atoms at distance 2: ', atom_distance_2)
                # print('Atoms connected to atom a: ',atom_distance_2_with_a)
                # print('Atoms connected to atom b: ',atom_distance_2_with_b)


                # Find atoms at atom distance 3
                atoms_to_avoid_3 = []
                atoms_to_avoid_3_with_a = []
                atoms_to_avoid_3_with_b = []
                
                atom_distance_3 = []
                atom_distance_3_with_a = []
                atom_distance_3_with_b = []

                atoms_to_avoid_3.extend(atom_distance_2)
                atoms_to_avoid_3.append(pd_atom+1)
                atoms_to_avoid_3.append(new_ligand_atoms_label[0])
                atoms_to_avoid_3.append(new_ligand_atoms_label[1])
                
                atoms_to_avoid_3_with_a.append(pd_atom+1)
                atoms_to_avoid_3_with_a.append(new_ligand_atoms_label[0])
                atoms_to_avoid_3_with_a.append(new_ligand_atoms_label[1])
                atoms_to_avoid_3_with_a.extend(atom_distance_2_with_a)
                
                atoms_to_avoid_3_with_b.append(pd_atom+1)
                atoms_to_avoid_3_with_b.append(new_ligand_atoms_label[0])
                atoms_to_avoid_3_with_b.append(new_ligand_atoms_label[1])
                atoms_to_avoid_3_with_b.extend(atom_distance_2_with_b)
                            
                
                # print('Atoms to avoid at distance 3: ',atoms_to_avoid_3)
                # print('Atoms to avoid at distance 3 with a: ',atoms_to_avoid_3_with_a)
                # print('Atoms to avoid at distance 3 with b: ',atoms_to_avoid_3_with_b)
                
                for atom in atom_distance_2:
                    atom_row = xyz_df.loc[atom-1]

                    atom_x = atom_row['X']
                    atom_y = atom_row['Y']
                    atom_z = atom_row['Z']

                    for index, row in xyz_df.iterrows():
                        atom_distance_diff_x = float(atom_x) - float(row['X'])
                        atom_distance_diff_y = float(atom_y) - float(row['Y'])
                        atom_distance_diff_z = float(atom_z) - float(row['Z'])

                        cart_distance_atom = np.sqrt(atom_distance_diff_x**2 + atom_distance_diff_y**2 + atom_distance_diff_z**2)
                        if 0.1 < cart_distance_atom < 1.93:
                            atom_distance_3.append(index+1)     
                
                for atom in atom_distance_2_with_a:
                    atom_row = xyz_df.loc[atom-1]

                    atom_x = atom_row['X']
                    atom_y = atom_row['Y']
                    atom_z = atom_row['Z']

                    for index, row in xyz_df.iterrows():
                        atom_distance_diff_x = float(atom_x) - float(row['X'])
                        atom_distance_diff_y = float(atom_y) - float(row['Y'])
                        atom_distance_diff_z = float(atom_z) - float(row['Z'])

                        cart_distance_atom = np.sqrt(atom_distance_diff_x**2 + atom_distance_diff_y**2 + atom_distance_diff_z**2)
                        if 0.1 < cart_distance_atom < 1.93:
                            atom_distance_3_with_a.append(index+1)      
                
                for atom in atom_distance_2_with_b:
                    atom_row = xyz_df.loc[atom-1]

                    atom_x = atom_row['X']
                    atom_y = atom_row['Y']
                    atom_z = atom_row['Z']

                    for index, row in xyz_df.iterrows():
                        atom_distance_diff_x = float(atom_x) - float(row['X'])
                        atom_distance_diff_y = float(atom_y) - float(row['Y'])
                        atom_distance_diff_z = float(atom_z) - float(row['Z'])

                        cart_distance_atom = np.sqrt(atom_distance_diff_x**2 + atom_distance_diff_y**2 + atom_distance_diff_z**2)
                        if 0.1 < cart_distance_atom < 1.93:
                            atom_distance_3_with_b.append(index+1)      
                

                #Remove duplicates
                atom_distance_3 = list(set(atom_distance_3))
                atom_distance_3 = [x for x in atom_distance_3 if x not in atoms_to_avoid_3]
                # print('Atoms at distance 3: ', atom_distance_3)

                atom_distance_3_with_a = list(set(atom_distance_3_with_a))
                atom_distance_3_with_a = [x for x in atom_distance_3_with_a if x not in atoms_to_avoid_3_with_a]
                # print('Atoms at distance 3 from donor atom A: ', atom_distance_3_with_a)

                atom_distance_3_with_b = list(set(atom_distance_3_with_b))
                # print('Atom 3, b: ', atom_distance_3_with_b)
                atom_distance_3_with_b = [x for x in atom_distance_3_with_b if x not in atoms_to_avoid_3_with_b]
                # print('Atoms at distance 3 from donor atom B: ', atom_distance_3_with_b)
                                    
                #Find atom at distance 4

                atom_distance_4 = []
                atoms_to_avoid_4 = []
                atom_distance_4_with_a = []
                atoms_to_avoid_4_with_a = []
                atom_distance_4_with_b = []
                atoms_to_avoid_4_with_b = []  
                
                atoms_to_avoid_4 = atoms_to_avoid_3
                atoms_to_avoid_4.extend(atom_distance_3)
                
                atoms_to_avoid_4_with_a = atoms_to_avoid_3_with_a
                atoms_to_avoid_4_with_a.extend(atom_distance_3_with_a)
                
                atoms_to_avoid_4_with_b = atoms_to_avoid_3_with_b
                atoms_to_avoid_4_with_b.extend(atom_distance_3_with_b)                
                
                           
                # print('Atoms to avoid at distance 4: ',atoms_to_avoid_4)
                # print('Atoms to avoid at distance 4 from a: ',atoms_to_avoid_4_with_a)
                # print('Atoms to avoid at distance 4 from b: ',atoms_to_avoid_4_with_b)
                
                for atom in atom_distance_3:
                    atom_row = xyz_df.loc[atom-1]

                    atom_x = atom_row['X']
                    atom_y = atom_row['Y']
                    atom_z = atom_row['Z']

                    for index, row in xyz_df.iterrows():
                        atom_distance_diff_x = float(atom_x) - float(row['X'])
                        atom_distance_diff_y = float(atom_y) - float(row['Y'])
                        atom_distance_diff_z = float(atom_z) - float(row['Z'])

                        cart_distance_atom = np.sqrt(atom_distance_diff_x**2 + atom_distance_diff_y**2 + atom_distance_diff_z**2)
                        if 0.1 < cart_distance_atom < 1.93:
                            atom_distance_4.append(index+1)     
                            
                for atom in atom_distance_3_with_a:
                    atom_row = xyz_df.loc[atom-1]

                    atom_x = atom_row['X']
                    atom_y = atom_row['Y']
                    atom_z = atom_row['Z']

                    for index, row in xyz_df.iterrows():
                        atom_distance_diff_x = float(atom_x) - float(row['X'])
                        atom_distance_diff_y = float(atom_y) - float(row['Y'])
                        atom_distance_diff_z = float(atom_z) - float(row['Z'])

                        cart_distance_atom = np.sqrt(atom_distance_diff_x**2 + atom_distance_diff_y**2 + atom_distance_diff_z**2)
                        if 0.1 < cart_distance_atom < 1.93:
                            atom_distance_4_with_a.append(index+1)                                 
                            
                for atom in atom_distance_3_with_b:
                    atom_row = xyz_df.loc[atom-1]

                    atom_x = atom_row['X']
                    atom_y = atom_row['Y']
                    atom_z = atom_row['Z']

                    for index, row in xyz_df.iterrows():
                        atom_distance_diff_x = float(atom_x) - float(row['X'])
                        atom_distance_diff_y = float(atom_y) - float(row['Y'])
                        atom_distance_diff_z = float(atom_z) - float(row['Z'])

                        cart_distance_atom = np.sqrt(atom_distance_diff_x**2 + atom_distance_diff_y**2 + atom_distance_diff_z**2)
                        if 0.1 < cart_distance_atom < 1.93:
                            atom_distance_4_with_b.append(index+1)                                             
                            
                
                #Remove duplicates
                atom_distance_4 = list(set(atom_distance_4))
                atom_distance_4 = [x for x in atom_distance_4 if x not in atoms_to_avoid_4]
                # print('Atoms at distance 4: ', atom_distance_4)

                atom_distance_4_with_a = list(set(atom_distance_4_with_a))
                atom_distance_4_with_a = [x for x in atom_distance_4_with_a if x not in atoms_to_avoid_4_with_a]
                # print('Atoms at distance 4 from A: ', atom_distance_4_with_a)
                
                atom_distance_4_with_b = list(set(atom_distance_4_with_b))
                atom_distance_4_with_b = [x for x in atom_distance_4_with_b if x not in atoms_to_avoid_4_with_b]
                # print('Atoms at distance 4 from B: ', atom_distance_4_with_b)
               
                              
                # Add 1 to every index to fit label naming
                names.append(name)
                methyl_atom_1.append(methyl_atoms[0]+1)
                methyl_atom_2.append(methyl_atoms[1]+1)
                hydrogens_methyl_1.append(hydrogen_atoms[0])
                hydrogens_methyl_2.append(hydrogen_atoms[1])
                ligand_atom_1.append(new_ligand_atoms_label[0])
                ligand_atom_2.append(new_ligand_atoms_label[1])
                pd_atoms.append(pd_atom+1)
                atom_distance_lists_2.append(atom_distance_2)
                atom_distance_lists_3.append(atom_distance_3)
                atom_distance_lists_4.append(atom_distance_4)
                atom_distance_lists_2_a.append(atom_distance_2_with_a)
                atom_distance_lists_3_a.append(atom_distance_3_with_a)
                atom_distance_lists_4_a.append(atom_distance_4_with_a)
                atom_distance_lists_2_b.append(atom_distance_2_with_b)                
                atom_distance_lists_3_b.append(atom_distance_3_with_b)                   
                atom_distance_lists_4_b.append(atom_distance_4_with_b)                   

                # Generate ligand_spe file by removing palladium and the two methyl atoms
                atoms_to_remove = methyl_atoms + hydrogen_indices[0] + hydrogen_indices[1]
                atoms_to_remove.append(pd_atom)
                # print('atoms_to_remove',atoms_to_remove)
                ligand_xyz_df = xyz_df.drop(atoms_to_remove,axis =0)
                # new_row = pd.DataFrame({'Atom':'filler', 'X':'filler', 'Y':'filler', 'Z':'filler'}, index=[0])
                # ligand_xyz_df = pd.concat([new_row,ligand_xyz_df.loc[:]]).reset_index(drop=True) 
                #Add first line with number of atoms extra filler line to fit format for graph generation
                natoms_ligand = ligand_xyz_df.shape[0]
                new_row = pd.DataFrame({'Atom':'filler', 'X':'filler', 'Y':'filler', 'Z':'filler'}, index=[0])
                ligand_xyz_df = pd.concat([new_row,ligand_xyz_df.loc[:]]).reset_index(drop=True) 
                new_row_2 = pd.DataFrame({'Atom':natoms_ligand, 'X':'', 'Y':'', 'Z':''}, index=[0])
                ligand_xyz_df = pd.concat([new_row_2,ligand_xyz_df.loc[:]]).reset_index(drop=True)
                ligand_xyz_filename = name+'-ligand_xyz.txt'

                # Ensure the output directory exists, create it if it doesn't
                os.makedirs(ligand_xyz_directory, exist_ok=True)

                # Combine the directory and filename to create the full file path
                ligand_xyz_file_path = os.path.join(ligand_xyz_directory, ligand_xyz_filename)
                print(ligand_xyz_file_path)

                # Save as text file in the ligand_xyz directory
                ligand_xyz_df.to_csv(ligand_xyz_file_path, header=False,index=False,sep=" ")

                #Use ligand_xyz_df to generate a graph to find the atoms for calculating internal angle
                # Using Graph network
                # Identify nodes with xyz coordinates as ID
                # Use removed PdMe2 structure to construct graph
                #Identify atom A and atom B nodes with their xyz coordinates
                # identify shortest path from A --> 
                # store the xyz atoms for these atoms, convert to atom label in the complex from by matching
                # do this code separately to the csv data generator and add into dataframe
                from plotly.offline import offline
                import networkx as nx
                from xyz2graph import MolGraph, to_networkx_graph, to_plotly_figure
                # Create the MolGraph object
                mg = MolGraph()

                # # Read the data from the .xyz file in the directory
                # mg.read_xyz(name+'-ligand_xyz.txt')

                # Read the data from the .xyz file in the directory
                mg.read_xyz(ligand_xyz_file_path)

                # Create the Plotly figure object
                fig = to_plotly_figure(mg)

                # Plot the figure
                # offline.plot(fig)

                # Convert the molecular graph to the NetworkX graph
                G = to_networkx_graph(mg)
                #Set donor atom xyz coordinates for matching
                
                atom_a_x = float(ligand_atom_1_x)
                atom_a_y = float(ligand_atom_1_y)
                atom_a_z = float(ligand_atom_1_z)
                atom_a_xyz = (atom_a_x,atom_a_y,atom_a_z)
                atom_b_x = float(ligand_atom_2_x)
                atom_b_y = float(ligand_atom_2_y)
                atom_b_z = float(ligand_atom_2_z)
                atom_b_xyz = (atom_b_x,atom_b_y,atom_b_z)

                starting_node = None
                ending_node = None

                starting_node_value = atom_a_xyz
                ending_node_value = atom_b_xyz

                # Iterate through nodes and find the one with the desired attribute value
                desired_node = None
                for node in G.nodes():
                    # print(G.nodes[node]['xyz'])
                    if G.nodes[node]['xyz'] == starting_node_value:
                        starting_node = node
                        print('Starting node:',node)
                    if G.nodes[node]['xyz'] == ending_node_value:
                        ending_node = node
                        print('Ending node:',node)

                if starting_node is None or ending_node is None:
                    raise ValueError("Could not find matching node(s) for the provided coordinates")                
                    
                shortest_path_xyz_list = []
                shortest_path_list = nx.shortest_path(G, source=starting_node, target=ending_node)
                for node in shortest_path_list:
                    print(G.nodes[node]['xyz'], ' ', node)                
                    shortest_path_xyz_list.append(G.nodes[node]['xyz'])
                #Create dictionary to store xyz for each ligand, then use it to match with the xyz in the original opt file
                
                shortest_path_xyz_dict = {}
                atom_order = 0
                for xyz in shortest_path_xyz_list:
                    shortest_path_atom = "Atom%d" %atom_order
                    shortest_path_xyz_dict[shortest_path_atom] = xyz
                    atom_order= atom_order + 1
                # print(shortest_path_xyz_dict)
                
                
                shortest_path_list_atom_label = [pd_atom+1]
                for xyz in shortest_path_xyz_dict:                       #Match with target xyz 
                    target_x = shortest_path_xyz_dict[xyz][0]
                    target_y = shortest_path_xyz_dict[xyz][1]
                    target_z = shortest_path_xyz_dict[xyz][2]
                    
                    # print(target_x)
                    # print(target_y)
                    # print(target_z)
                    
                    for index, row in xyz_df.iterrows():
                        if float(target_x) == float(row['X']) and (float(target_y) == float(row['Y']) and float(target_z) == float(row['Z'])):
                            shortest_path_list_atom_label.append(index+1)
                            break
                            
                
                # print('Atom label shortest path: ', shortest_path_list_atom_label)
                shortest_path_atom_label_lists.append(shortest_path_list_atom_label)
                              

data = {
    'Filename' : names,
    'Pd_atom label' : pd_atoms,
    'Methyl_atom_1 label' : methyl_atom_1,
    'Methyl_atom_2 label' : methyl_atom_2,
    'Hydrogens_methyl_1_label' : hydrogens_methyl_1,
    'Hydrogens_methyl_2_label' : hydrogens_methyl_2,
    'Ligand_atom_1 label' : ligand_atom_1,
    'Ligand_atom_2 label' : ligand_atom_2,
    'Atoms_distance_2 label' : atom_distance_lists_2,
    'Atoms_distance_3 label' : atom_distance_lists_3,
    'Atoms_distance_4 label' : atom_distance_lists_4,
    'Atoms_distance_2_a label': atom_distance_lists_2_a,
    'Atoms_distance_3_a label': atom_distance_lists_3_a,
    'Atoms_distance_4_a label': atom_distance_lists_4_a,
    'Atoms_distance_2_b label': atom_distance_lists_2_b,
    'Atoms_distance_3_b label': atom_distance_lists_3_b,
    'Atoms_distance_4_b label': atom_distance_lists_4_b,
    'Atom label shortest paths' : shortest_path_atom_label_lists
    }


atom_label_df = pd.DataFrame(data)


# Modify Atom_label_df to only include those that are have all three files 

mask = ~atom_label_df['Filename'].isin(missing_list)
atom_label_df = atom_label_df[mask]


csv_filename = 'atoms_label.csv'
atom_label_df.to_csv(csv_filename, index=False)
print(atom_label_df.head())

ligand-001-n-n-1
Pd number index:  1
['C', 'Pd', 'C', 'H', 'H', 'H', 'C', 'H', 'H', 'H', 'H', 'H', 'C', 'H', 'C', 'H', 'C', 'C', 'C', 'H', 'H', 'N', 'N', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H']
Ligand atom label:  22  Ligand atom B nbo:  -0.55126
Ligand atom label:  23  Ligand atom A nbo:  -0.48882
Atom_a:  23 Atom_b:  22
C:\Users\George\Desktop\Research_UCLA\CIC\Computational Work\Project cluster version 2\project_catalyst_repurposing\structures_ligand_id\pd_me2_structures\ligand_xyz\ligand-001-n-n-1-ligand_xyz.txt


C:\Users\George\AppData\Local\Temp\ipykernel_7832\386804788.py:583: DeprecationWarning:

to_plotly_figure is deprecated and will be removed in version 4.0.0. Use MolGraph.to_plotly() instead.

C:\Users\George\AppData\Local\Temp\ipykernel_7832\386804788.py:589: DeprecationWarning:

to_networkx_graph is deprecated and will be removed in version 4.0.0. Use MolGraph.to_networkx() instead.



Ending node: 12
Starting node: 13
(-1.491218, -0.956693, 0.003991)   13
(-0.668134, -2.116302, 0.349288)   9
(0.678885, -2.114263, -0.344581)   0
(1.496369, -0.949341, -0.00377)   12
ligand-002-p-p-1
Pd number index:  1
['C', 'Pd', 'C', 'H', 'H', 'H', 'C', 'H', 'H', 'H', 'H', 'H', 'C', 'H', 'H', 'P', 'P', 'C', 'C', 'C', 'C', 'H', 'C', 'H', 'C', 'H', 'H', 'H', 'C', 'C', 'C', 'C', 'H', 'C', 'H', 'C', 'H', 'H', 'H', 'C', 'C', 'C', 'C', 'H', 'C', 'H', 'C', 'H', 'H', 'H', 'C', 'C', 'C', 'C', 'H', 'C', 'H', 'C', 'H', 'H', 'H']
Ligand atom label:  16  Ligand atom A nbo:  1.17518
Ligand atom label:  17  Ligand atom B nbo:  0.95542
Atom_a:  16 Atom_b:  17
C:\Users\George\Desktop\Research_UCLA\CIC\Computational Work\Project cluster version 2\project_catalyst_repurposing\structures_ligand_id\pd_me2_structures\ligand_xyz\ligand-002-p-p-1-ligand_xyz.txt


C:\Users\George\AppData\Local\Temp\ipykernel_7832\386804788.py:583: DeprecationWarning:

to_plotly_figure is deprecated and will be removed in version 4.0.0. Use MolGraph.to_plotly() instead.

C:\Users\George\AppData\Local\Temp\ipykernel_7832\386804788.py:589: DeprecationWarning:

to_networkx_graph is deprecated and will be removed in version 4.0.0. Use MolGraph.to_networkx() instead.



Starting node: 6
Ending node: 7
(1.688751, -0.141338, 0.220698)   6
(0.91271, -0.573164, 1.865196)   0
(-0.509342, -0.025928, 1.940818)   3
(-1.432395, -0.288589, 0.318623)   7
ligand-003-p-p-2
Pd number index:  2
['P', 'C', 'Pd', 'C', 'H', 'H', 'H', 'C', 'H', 'H', 'H', 'P', 'H', 'H', 'C', 'H', 'C', 'H', 'H', 'H', 'C', 'C', 'C', 'C', 'H', 'C', 'H', 'C', 'H', 'H', 'H', 'C', 'C', 'C', 'C', 'H', 'C', 'H', 'C', 'H', 'H', 'H', 'C', 'C', 'C', 'C', 'H', 'C', 'H', 'C', 'H', 'H', 'H', 'C', 'C', 'C', 'C', 'H', 'C', 'H', 'C', 'H', 'H', 'H']
Ligand atom label:  1  Ligand atom A nbo:  1.02659
Ligand atom label:  12  Ligand atom B nbo:  1.02437
Atom_a:  1 Atom_b:  12
C:\Users\George\Desktop\Research_UCLA\CIC\Computational Work\Project cluster version 2\project_catalyst_repurposing\structures_ligand_id\pd_me2_structures\ligand_xyz\ligand-003-p-p-2-ligand_xyz.txt
Starting node: 0
Ending node: 2
(-1.70582, 0.264426, -0.437921)   0
(-1.325801, 0.370988, -2.256722)   7
(-0.054687, -0.3512, -2.695941)   5

...

Pd number index:  52
['P', 'O', 'P', 'N', 'C', 'C', 'C', 'C', 'C', 'C', 'H', 'H', 'H', 'H', 'H', 'C', 'C', 'C', 'C', 'C', 'C', 'H', 'H', 'H', 'H', 'H', 'C', 'C', 'C', 'C', 'H', 'C', 'H', 'C', 'H', 'H', 'C', 'C', 'C', 'C', 'C', 'C', 'H', 'H', 'C', 'C', 'C', 'C', 'C', 'C', 'H', 'H', 'Pd', 'H', 'H', 'C', 'H', 'H', 'H', 'C', 'H', 'H', 'H', 'H', 'C', 'H', 'H', 'H', 'H', 'N', 'C', 'H', 'H', 'H', 'O', 'O', 'C', 'H', 'H', 'H', 'C', 'H', 'H', 'H']
Ligand atom label:  1  Ligand atom B nbo:  1.21295
Ligand atom label:  2  Ligand atom A nbo:  -0.94405
Atom_a:  1 Atom_b:  2
C:\Users\George\Desktop\Research_UCLA\CIC\Computational Work\Project cluster version 2\project_catalyst_repurposing\structures_ligand_id\pd_me2_structures\ligand_xyz\ligand-531-p-o-182-ligand_xyz.txt
Starting node: 0
Ending node: 1
(-1.378344, -0.151407, 0.054044)   0
(0.344155, -0.303648, -0.313973)   3
(0.85226, -1.741235, -1.071464)   2
(-0.324865, -2.470015, -1.67574)   1
ligand-532-p-o-183


C:\Users\George\AppData\Local\Temp\ipykernel_7832\386804788.py:583: DeprecationWarning:

to_plotly_figure is deprecated and will be removed in version 4.0.0. Use MolGraph.to_plotly() instead.

C:\Users\George\AppData\Local\Temp\ipykernel_7832\386804788.py:589: DeprecationWarning:

to_networkx_graph is deprecated and will be removed in version 4.0.0. Use MolGraph.to_networkx() instead.



Pd number index:  52
['P', 'O', 'P', 'N', 'C', 'C', 'C', 'C', 'C', 'C', 'H', 'H', 'H', 'H', 'H', 'C', 'C', 'C', 'C', 'C', 'C', 'H', 'H', 'H', 'H', 'H', 'C', 'C', 'C', 'C', 'H', 'C', 'H', 'C', 'H', 'H', 'C', 'C', 'C', 'C', 'C', 'C', 'H', 'H', 'C', 'C', 'C', 'C', 'C', 'C', 'H', 'H', 'Pd', 'H', 'H', 'C', 'H', 'H', 'H', 'C', 'H', 'H', 'H', 'H', 'C', 'H', 'H', 'H', 'C', 'H', 'C', 'F', 'F', 'F', 'F', 'F', 'F', 'N', 'C', 'H', 'H', 'H']
Ligand atom label:  1  Ligand atom B nbo:  1.2117
Ligand atom label:  2  Ligand atom A nbo:  -0.94392
Atom_a:  1 Atom_b:  2
C:\Users\George\Desktop\Research_UCLA\CIC\Computational Work\Project cluster version 2\project_catalyst_repurposing\structures_ligand_id\pd_me2_structures\ligand_xyz\ligand-532-p-o-183-ligand_xyz.txt
Starting node: 0
Ending node: 1
(-0.729091, 0.06092, -1.133668)   0
(-0.122904, 0.758147, 0.364903)   3
(-0.350841, 2.443689, 0.559089)   2
(-1.39351, 2.921349, -0.423345)   1
           Filename  Pd_atom label  Methyl_atom_1 label  Methyl_ato

C:\Users\George\AppData\Local\Temp\ipykernel_7832\386804788.py:583: DeprecationWarning:

to_plotly_figure is deprecated and will be removed in version 4.0.0. Use MolGraph.to_plotly() instead.

C:\Users\George\AppData\Local\Temp\ipykernel_7832\386804788.py:589: DeprecationWarning:

to_networkx_graph is deprecated and will be removed in version 4.0.0. Use MolGraph.to_networkx() instead.



(-0.453569, -0.001281, 0.025399)
Starting node: 0
(0.280081, -1.353772, -1.005911)
(-0.921593, -0.840281, 1.590733)
(-2.103593, 0.197254, -0.783083)
(1.668692, -1.575559, -0.962845)
(-0.490093, -2.13391, -1.877283)
(2.747601, -0.627605, 0.149301)
(2.255078, -2.554863, -1.759459)
(2.689225, 0.772553, -0.390403)
Ending node: 4
(2.087469, -0.773044, 1.459512)
(4.077666, -1.229666, 0.013235)
(-1.23878, -0.01711, 2.676207)
(-0.994024, -2.227244, 1.740577)
(-3.294065, -0.363605, -0.309618)
(-2.143058, 0.98824, -1.939927)
(1.473211, -3.326043, -2.617489)
(3.339564, -2.691768, -1.683251)
(0.098058, -3.115212, -2.675416)
(1.941613, -4.095451, -3.244141)
(-0.527882, -3.716914, -3.346207)
(-1.574857, -1.97574, -1.934228)
(-4.495711, -0.15301, -0.988708)
(-3.286777, -0.976056, 0.601348)
(-4.520151, 0.616713, -2.149301)
(-5.422156, -0.597717, -0.604099)
(-3.33807, 1.187924, -2.624157)
(-5.46501, 0.78146, -2.682079)
(-3.35054, 1.8065, -3.529898)
(-1.214294, 1.461113, -2.290573)
(-1.654843, -0.572951

## Convert all opt .out files into spe .com files from opt_directory to spe_complex_directory

In [8]:
xyz_match = ['X           Y           Z']
for subdir,dirs,files in os.walk(opt_directory):                  # Loop over each directory, subdirectory and files
    for file in files:                                      # Loop over each file
        if any([file.endswith('-opt.out')]):                    # If file is a .out file
            filename = os.path.join(subdir, file)       # Return path to file
            name = Path(filename).stem.replace('-opt',"")         # Extract filename from the end of path and return as a string
            print(name)
        
            
            mylines = []
            with open (filename, 'rt') as myfile:       # Open .out for reading text
                # myfile = myfile.read()                # Read the entire file to a string
                for myline in myfile:                    # For each line, stored as myline,
                    mylines.append(myline)               # add its contents to mylines list.


                #initialize charge and multiplicity
                charge = None
                multiplicity = None  
                
                # Find and extract the Charge and Multiplicity values

                for line in mylines:
                    if 'Charge =' in line and 'Multiplicity =' in line:
                        # Use regular expressions to extract numbers
                        charge_multiplicity = re.findall(r'Charge\s*=\s*(-?\d+)\s*Multiplicity\s*=\s*(\d+)', line)
                        charge_str, multiplicity_str = charge_multiplicity[0]
                        charge = int(charge_str)
                        print('Charge: ',charge)
                        multiplicity = int(multiplicity_str)
                        print('Multiplicity: ',multiplicity)
                

#                 # Find XYZ Coordinates

                for line in mylines:
                    if 'NAtoms=' in line:
                        number_list = re.findall('-?\d*\.?\d+',line)            # get NAtoms value
                        natoms = int(number_list[0])
#                         print(natoms)                
                
                xyz_count = 0
                for line in mylines:
                    for phrase in xyz_match:                                # iterate through each phrases
                        if phrase in line:                                          # check if phrase is in line
                            xyz_count = xyz_count + 1
                
                line_count = 0
                for line in mylines:
                    line_count = line_count + 1
                    for phrase in xyz_match:                                # iterate through each phrases
                        if phrase in line:                                          # check if phrase is in line
                            xyz_count = xyz_count - 1
                            if xyz_count > 0:
                                continue
                            elif xyz_count == 0:
                                
                                                                               # For loop for generating the XYZ coordinates
                                count = 0
                                xyz = []
                                while count < natoms:
                                    count = count + 1
                                    xyz.append(mylines[line_count + 1])
                                    line_count = line_count +1


                x_coord = []
                y_coord = []
                z_coord = []
                atom_symbol =[]
        
                # Generate XYZ file in .txt form, then find xyz coordinates for metal, atom_a and atom_b
                for line in xyz:
                    number_list = re.findall('-?\d*\.?\d+',line)
                    # print(number_list)
                    # print(number_list[0])
                    atom_number = int(number_list[0])
                    element_number = int(number_list[1])
                    atom_x = float(number_list[3])
                    atom_x = f"{atom_x:.6f}"
                    atom_y = float(number_list[4])
                    atom_y = f"{atom_y:.6f}"
                    atom_z = float(number_list[5])
                    atom_z = f"{atom_z:.6f}"
                
                    # # Make xyz coord into .txt file 
                    # x_coord.append(atom_x)
                    # y_coord.append(atom_y)
                    # z_coord.append(atom_z)
                    # atom_symbol.append(element[int(element_number)])
                    # unique_atoms = list(set(atom_symbol))
                    # name_xyz = name + '-complex_xyz.txt'          

#                     # Make xyz coord into .txt file 
                    x_coord.append(atom_x)
                    y_coord.append(atom_y)
                    z_coord.append(atom_z)
                    atom_symbol.append(element_dict[str(element_number)])
                    unique_atoms = list(set(atom_symbol))
                    name_xyz = name + '-complex_xyz.txt'  
                    name_spe_xyz = name + '-complex_spe_xyz.txt'
                    
                # data = {
                #     'Atom': atom_symbol,
                #     'X': x_coord,
                #     'Y': y_coord,
                #     'Z': z_coord
                # }
                
                # xyz_df = pd.DataFrame(data)
                # xyz_df.to_csv(name_xyz, header=False, index=False, sep = " ")          # Generates .txt file 


                spe_data = {
                    'Name': name,
                    'Charge': charge,
                    'Multiplicity': multiplicity,
                    'Atom': atom_symbol,
                    'X': x_coord,
                    'Y': y_coord,
                    'Z': z_coord
                }
                xyz_spe_df = pd.DataFrame(spe_data).drop_duplicates()
                xyz_spe_df.to_csv(name_spe_xyz, header=False, index=False, sep = " ")          # Generates .txt file
                print(xyz_spe_df)

                os.makedirs(spe_complex_directory, exist_ok=True)


                # Extract unique elements from 'Atom' column, excluding 'Pd'
                elements = xyz_spe_df[xyz_spe_df['Atom'] != 'Pd']['Atom'].unique()
                
                # Convert the list of elements to a string formatted for output
                elements_string = " ".join(sorted(elements)) + " 0"
                                
                # Format the file content
                output_lines = []
                output_lines.append(f"%mem=16000MB\n")
                output_lines.append(f"%nprocshared=16\n")
                output_lines.append(f"%chk={name}-spe.chk\n")
                output_lines.append(f"#p m06/genecp pop=nbo\n\n")
                output_lines.append(f"{name}\n\n")
                output_lines.append(f"{charge} {multiplicity}\n")
                #Add the atom coordinates
                for index, row in xyz_spe_df.iterrows():
                    output_lines.append(f"{row['Atom']} {row['X']} {row['Y']} {row['Z']}\n")
                   
                # Add the additional lines with the dynamic elements_string
                output_lines.append("\n")
                output_lines.append(f"{elements_string}\n")
                output_lines.append("def2tzvp\n")
                output_lines.append("****\n")
                output_lines.append("Pd 0\n")
                output_lines.append("LANL2DZ\n")
                output_lines.append("****\n\n")
                output_lines.append("Pd 0\n")
                output_lines.append("LANL2DZ\n")
                output_lines.append("\n\n")
                # Specify the output path and save the file
                filename = f"{name}-spe.com"  # Replace with the desired filename or use 'name' variable
                output_path = os.path.join(spe_complex_directory, filename)
                
                # Save the output as a .txt file
                with open(output_path, 'w') as file:
                    file.writelines(output_lines)
                
                print(f"File saved to {output_path}")



ligand-001-n-n-1
Charge:  0
Multiplicity:  1
Charge:  0
Multiplicity:  1
                Name  Charge  Multiplicity Atom          X          Y  \
0   ligand-001-n-n-1       0             1    C   0.678885  -2.114263   
1   ligand-001-n-n-1       0             1   Pd  -0.001914   0.826905   
2   ligand-001-n-n-1       0             1    C  -1.379532   2.334521   
3   ligand-001-n-n-1       0             1    H  -1.549512   2.670152   
4   ligand-001-n-n-1       0             1    H  -2.335276   1.947701   
5   ligand-001-n-n-1       0             1    H  -1.064722   3.207270   
6   ligand-001-n-n-1       0             1    C   1.365704   2.343597   
7   ligand-001-n-n-1       0             1    H   2.331368   1.957365   
8   ligand-001-n-n-1       0             1    H   1.515839   2.700399   
9   ligand-001-n-n-1       0             1    H   1.053042   3.202010   
10  ligand-001-n-n-1       0             1    H   1.220897  -3.059859   
11  ligand-001-n-n-1       0             1    H   0

In [9]:
xyz_spe_df

,Name,Charge,Multiplicity,Atom,X,Y,Z
0,ligand-532-p-o-183,0,1,P,-0.729091,0.060920,-1.133668
1,ligand-532-p-o-183,0,1,O,-1.393510,2.921349,-0.423345
2,ligand-532-p-o-183,0,1,P,-0.350841,2.443689,0.559089
3,ligand-532-p-o-183,0,1,N,-0.122904,0.758147,0.364903
4,ligand-532-p-o-183,0,1,C,-0.887958,2.690993,2.268850
...,...,...,...,...,...,...,...
77,ligand-532-p-o-183,0,1,N,3.332827,-2.335491,3.587570
78,ligand-532-p-o-183,0,1,C,4.736450,-2.386720,3.262441
79,ligand-532-p-o-183,0,1,H,5.238188,-3.118016,3.915658
80,ligand-532-p-o-183,0,1,H,4.909615,-2.696278,2.210550


In [10]:
atom_label_df

,Filename,Pd_atom label,Methyl_atom_1 label,Methyl_atom_2 label,Hydrogens_methyl_1_label,Hydrogens_methyl_2_label,Ligand_atom_1 label,Ligand_atom_2 label,Atoms_distance_2 label,Atoms_distance_3 label,Atoms_distance_4 label,Atoms_distance_2_a label,Atoms_distance_3_a label,Atoms_distance_4_a label,Atoms_distance_2_b label,Atoms_distance_3_b label,Atoms_distance_4_b label,Atom label shortest paths
0,ligand-001-n-n-1,2,3,7,"[6, 4, 5]","[10, 9, 8]",23,22,"[1, 13, 15, 17, 18, 19]","[11, 12, 14, 16, 20, 21, 24, 25, 26, 27, 28, 2...",[],"[17, 18, 19]","[32, 33, 1, 20, 21, 28, 29, 30, 31]","[11, 12]","[1, 13, 15]","[11, 12, 14, 16, 19, 24, 25, 26, 27]","[20, 21]","[2, 23, 19, 1, 22]"
1,ligand-002-p-p-1,2,7,3,"[9, 8, 10]","[4, 6, 5]",16,17,"[1, 40, 13, 18, 51, 29]","[41, 42, 11, 12, 14, 15, 19, 20, 52, 53, 30, 31]","[21, 22, 23, 24, 32, 33, 34, 35, 43, 44, 45, 4...","[1, 18, 29]","[11, 12, 13, 19, 20, 30, 31]","[32, 33, 34, 35, 14, 15, 21, 22, 23, 24]","[13, 40, 51]","[1, 41, 42, 14, 15, 52, 53]","[11, 12, 43, 44, 45, 46, 54, 55, 56, 57]","[2, 16, 1, 13, 17]"
2,ligand-003-p-p-2,3,4,8,"[5, 7, 6]","[10, 11, 9]",1,12,"[32, 2, 43, 17, 21, 54]","[33, 34, 13, 14, 15, 44, 45, 18, 19, 55, 22, 2...","[16, 20, 24, 25, 26, 27, 35, 36, 37, 38, 46, 4...","[17, 43, 54]","[44, 45, 15, 18, 19, 55, 56]","[2, 46, 47, 48, 49, 16, 20, 57, 58, 59, 60]","[2, 21, 32]","[33, 34, 13, 14, 15, 22, 23]","[35, 36, 37, 38, 16, 17, 20, 24, 25, 26, 27]","[3, 1, 17, 15, 2, 12]"
3,ligand-004-p-p-3,3,8,4,"[11, 10, 9]","[6, 5, 7]",45,1,"[42, 46, 16, 20, 57, 31]","[32, 2, 33, 43, 44, 14, 47, 48, 17, 18, 21, 22...","[12, 13, 15, 19, 23, 24, 25, 26, 34, 35, 36, 3...","[42, 46, 57]","[2, 43, 44, 47, 48, 58, 59]","[12, 13, 14, 49, 50, 51, 52, 60, 61, 62, 63]","[16, 20, 31]","[32, 33, 14, 17, 18, 21, 22]","[34, 35, 36, 37, 2, 15, 19, 23, 24, 25, 26]","[3, 45, 42, 2, 14, 16, 1]"
4,ligand-005-c-n-1,1,42,46,"[43, 44, 45]","[49, 48, 47]",2,59,"[50, 52, 61, 6]","[3, 4, 7, 51, 55, 56, 25]","[5, 8, 9, 20, 53, 54, 24, 58]","[6, 61]","[3, 4, 7, 25]","[5, 8, 9, 50, 20, 24]","[50, 52]","[51, 55, 56, 25]","[6, 53, 54, 24, 58]","[1, 2, 6, 25, 50, 59]"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
485,ligand-528-p-o-179,53,56,60,"[57, 59, 58]","[61, 63, 62]",1,2,"[40, 50, 3, 4]","[5, 39, 41, 45, 49, 21, 27]","[38, 6, 70, 10, 42, 46, 79, 48, 16, 20, 54, 55...","[4, 40, 50]","[3, 39, 41, 45, 49, 27]","[5, 38, 70, 42, 46, 79, 48, 21, 54, 55, 28, 29]",[3],"[21, 4, 5]","[6, 10, 16, 20, 27]","[53, 1, 4, 3, 2]"
486,ligand-529-p-o-180,53,56,60,"[59, 57, 58]","[63, 61, 62]",1,2,"[40, 50, 3, 4]","[5, 39, 41, 45, 49, 21, 27]","[64, 69, 38, 6, 10, 42, 46, 48, 16, 20, 54, 55...","[4, 40, 50]","[3, 39, 41, 45, 49, 27]","[64, 5, 38, 69, 42, 46, 48, 21, 54, 55, 28, 29]",[3],"[21, 4, 5]","[6, 10, 16, 20, 27]","[53, 1, 4, 3, 2]"
487,ligand-530-p-o-181,53,60,56,"[63, 61, 62]","[59, 57, 58]",1,2,"[40, 50, 3, 4]","[5, 39, 41, 45, 49, 21, 27]","[38, 6, 10, 75, 42, 76, 46, 48, 16, 20, 54, 55...","[4, 40, 50]","[3, 39, 41, 45, 49, 27]","[5, 38, 42, 75, 76, 46, 48, 21, 54, 55, 28, 29]",[3],"[21, 4, 5]","[6, 10, 16, 20, 27]","[53, 1, 4, 3, 2]"
488,ligand-531-p-o-182,53,56,60,"[59, 57, 58]","[63, 61, 62]",1,2,"[40, 50, 3, 4]","[5, 39, 41, 45, 49, 21, 27]","[64, 69, 38, 6, 10, 42, 46, 48, 16, 20, 54, 55...","[4, 40, 50]","[3, 39, 41, 45, 49, 27]","[64, 5, 38, 69, 42, 46, 48, 21, 54, 55, 28, 29]",[3],"[21, 4, 5]","[6, 10, 16, 20, 27]","[53, 1, 4, 3, 2]"
